### Carga del Dataset e Inspección Inicial

In [2]:
import numpy as np
import pandas as pd

# Cargar dataset original
ruta_archivo = "../data/raw/inventario_ferreteria.csv"
df = pd.read_csv(ruta_archivo)

print(f"Dimensiones del dataset: {df.shape[0]} filas y {df.shape[1]} columnas\n")
df.head()

Dimensiones del dataset: 4280 filas y 11 columnas



,id_movimiento,fecha,almacen,provincia,producto,categoria,caduca,estado_ciclo,cantidad,costo_almacenamiento,fecha_vencimiento
0,1,2025-05-22 20:21:50,Sucursal Mendoza,Mendoza,Martillo carpintero,Herramientas manuales,False,Devuelto a proveedor,47.0,28.28,NaN
1,2,2024-10-29 18:09:17,Depósito Central,Buenos Aires,Alicate universal,Herramientas manuales,False,Devuelto cliente,37.0,35.50,NaN
2,3,2024-09-26 05:35:28,Sucursal Córdoba,Córdoba,Taladro eléctrico,Herramientas eléctricas,False,Devuelto a proveedor,49.0,54.37,NaN
3,4,2025-07-01 12:13:32,Sucursal Córdoba,Córdoba,Soldadora eléctrica,Herramientas eléctricas,False,Removido inventario,62.0,84.93,NaN
4,5,2025-01-01 07:55:59,Sucursal Córdoba,Córdoba,Caño galvanizado 1m,Plomería,False,Empacado,74.0,93.29,NaN


### Limpiar los Datos y Convertir los tipos

In [3]:
# Convertir las fechas
df["fecha"] = pd.to_datetime(df["fecha"], errors="coerce")
df["fecha_vencimiento"] = pd.to_datetime(df["fecha_vencimiento"], errors="coerce")

# Imputar valores nulos
df["cantidad"] = df["cantidad"].fillna(0)
df["costo_almacenamiento"] = df["costo_almacenamiento"].fillna(
    df.groupby("categoria")["costo_almacenamiento"].transform("median")
)
df["estado_ciclo"] = df["estado_ciclo"].fillna("No especificado")

# Guardar dataset limpio
df.to_csv("../data/processed/inventario_limpio.csv", index=False)
print("¡Limpieza completada y guardada en '../data/processed/inventario_limpio.csv'!")

¡Limpieza completada y guardada en '../data/processed/inventario_limpio.csv'!


### Crear Base de Datos SQLite
Cargar el dataset limpio en una base de datos relacional local

In [4]:
import sqlite3

# Crear conexión y base de datos local
conexion = sqlite3.connect("../data/inventario_ferreteria.db")

# Leer el CSV desde la subcarpeta correcta (ej. 'processed')
df_limpio = pd.read_csv("../data/processed/inventario_limpio.csv")

# Volcar el DataFrame limpio a una tabla SQL
df_limpio.to_sql(
    "movimientos_inventario", conexion, if_exists="replace", index=False
)

print("¡Datos guardados exitosamente en la base de datos!")

¡Datos guardados exitosamente en la base de datos!


### Consultas SQL y análisis logístico por ubicación

In [5]:
query_logistica = """
    SELECT 
        provincia,
        almacen,
        COUNT(id_movimiento) AS total_movimientos,
        SUM(cantidad) AS unidades_totales,
        ROUND(SUM(costo_almacenamiento), 2) AS costo_almacenamiento_total
    FROM movimientos_inventario
    GROUP BY provincia, almacen
    ORDER BY costo_almacenamiento_total DESC;
"""

pd.read_sql_query(query_logistica, conexion)

,provincia,almacen,total_movimientos,unidades_totales,costo_almacenamiento_total
0,Córdoba,Sucursal Córdoba,883,82415.0,101463.40
1,Buenos Aires,Depósito Central,867,82154.0,97541.24
2,Tucumán,Sucursal Tucumán,836,79472.0,96957.52
3,Santa Fe,Sucursal Rosario,868,77194.0,93071.71
4,Mendoza,Sucursal Mendoza,826,74343.0,92287.54


### Categorización ABC del inventario

In [6]:
# Consulta SQL para extraer costos por producto
query_abc = """
    SELECT 
        producto,
        categoria,
        COUNT(id_movimiento) AS movimientos_totales,
        SUM(cantidad) AS unidades_acumuladas,
        SUM(costo_almacenamiento) AS costo_total_almacenamiento
    FROM movimientos_inventario
    GROUP BY producto, categoria
    ORDER BY costo_total_almacenamiento DESC;
"""

abc_df = pd.read_sql_query(query_abc, conexion)

# Calcular porcentaje acumulado y clasificación ABC
abc_df["porcentaje_costo"] = (
    abc_df["costo_total_almacenamiento"].cumsum()
    / abc_df["costo_total_almacenamiento"].sum()
) * 100


def clasificar_abc(porcentaje):
  if porcentaje <= 80:
    return "Clase A"
  elif porcentaje <= 95:
    return "Clase B"
  else:
    return "Clase C"


abc_df["clasificacion_abc"] = abc_df["porcentaje_costo"].apply(clasificar_abc)

print("--- Resumen de Clasificación ABC ---")
print(abc_df["clasificacion_abc"].value_counts())
abc_df.head(10)

--- Resumen de Clasificación ABC ---
clasificacion_abc
Clase A    31
Clase B    12
Clase C     7
Name: count, dtype: int64


,producto,categoria,movimientos_totales,unidades_acumuladas,costo_total_almacenamiento,porcentaje_costo,clasificacion_abc
0,Tarugos plásticos,Fijaciones,95,17865.0,21117.270,4.387353,Clase A
1,Pernos con tuerca,Fijaciones,96,18958.0,20366.710,8.618769,Clase A
2,Caja de tornillos,Fijaciones,94,18499.0,19886.920,12.750503,Clase A
3,Clavos de acero,Fijaciones,86,16993.0,19699.290,16.843254,Clase A
4,Arena fina m3,Materiales construcción,87,16474.0,19141.830,20.820187,Clase A
5,Arandelas metálicas,Fijaciones,81,16069.0,18604.880,24.685563,Clase A
6,Caño PVC 2m,Materiales construcción,82,14673.0,17464.100,28.313929,Clase A
7,Bloques de hormigón,Materiales construcción,86,16481.0,17209.395,31.889376,Clase A
8,Yeso en polvo 30kg,Materiales construcción,82,16010.0,15262.005,35.060231,Clase A
9,Bolsa de cemento 50kg,Materiales construcción,66,12023.0,13803.345,37.928033,Clase A
